In [6]:
import torch

torch.__version__

'2.14.0+cu130'

In [2]:
from torchvision import transforms

transform_dict = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(32, (0.8, 1.0)),
        transforms.RandomAffine(10, (0.1, 0.1), (1.0, 1.0), shear=5),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616)
        )
    ]),
    "val": transforms.Compose([
        transforms.ToTensor(),

        transforms.Normalize(
            (0.4914, 0.4822, 0.4465),
            (0.2470, 0.2435, 0.2616)
        )
    ])
}

In [3]:
from torchvision.datasets import CIFAR10
from torchvision import transforms

transform = transforms.ToTensor()

train_dataset = CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform_dict["train"]
)

test_dataset = CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform_dict["val"]
)

print(len(train_dataset), len(test_dataset), type(train_dataset))

50000 10000 <class 'torchvision.datasets.cifar.CIFAR10'>


In [7]:
from torch.utils.data import DataLoader

generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    dataset=train_dataset,
    shuffle=True,
    generator=generator,
    batch_size=128,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
)

test_loader = DataLoader(
    dataset=test_dataset,
    shuffle=False,
    generator=generator,
    batch_size=128,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
)

In [9]:
from torch import nn

class MyCNN1(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.backbone = nn.Sequential(
            # b, 3, 32, 32
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # b, 32, 16, 16
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # b, 64, 8, 8
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # b, 64, 8, 8
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # b, 32, 8, 8
            nn.Conv2d(32, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )

        # 16*8*8
        self.flatten = nn.Flatten()

        self.features = nn.Sequential(
            nn.Linear(16*8*8, 32),
            nn.ReLU(),

            nn.Linear(32, 32),
            nn.ReLU(),

            nn.Linear(32, num_classes)

        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.flatten(x)
        return self.features(x)

In [10]:
class MyCNN2(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.backbone = nn.Sequential(
            # [B, 3, 32, 32]
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),
            # [B, 32, 16, 16]


            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),
            # [B, 64, 8, 8]


            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2),
            # [B, 128, 4, 4]
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        # [B, 128, 1, 1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # [B, 128]

            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = self.classifier(x)

        return x

In [11]:
def run_epoch(model, data_loader, criterion, optimizer=None):
    is_training =  False if optimizer is None else True

    total_loss = 0
    total_correct = 0
    total_count = 0

    device = next(model.parameters()).device

    if not is_training:
        with torch.inference_mode():
            model.eval()
            for images, labels in data_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                logits = model(images)

                loss = criterion(logits, labels)

                total_count += labels.size(0)
                total_loss += loss.item() * labels.size(0)
                total_correct += (logits.argmax(dim=1) == labels).sum().item()
    else:
        model.train()
        for images, labels in data_loader:
             images = images.to(device, non_blocking=True)
             labels = labels.to(device, non_blocking=True)

             optimizer.zero_grad()

             logits = model(images)
             loss = criterion(logits, labels)

             loss.backward()
             optimizer.step()

             total_count += labels.size(0)
             total_loss += loss.item() * labels.size(0)
             total_correct += (logits.argmax(dim=1) == labels).sum().item()

    return (total_loss / total_count), (total_correct / total_count)


In [12]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MyCNN2(10).to(DEVICE)

criterion = nn.CrossEntropyLoss()

# optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.001)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.001)


In [13]:
EPOCH = 50

for epoch in range(1, EPOCH+1):
    # 예측하기
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)

    val_loss, val_acc = run_epoch(model, test_loader, criterion)

    if epoch % 10 == 0:
        print(f"epoch: {epoch} | train loss: {train_loss} train_acc: {train_acc} | val_loss: {val_loss} val_acc: {val_acc} ")

epoch: 10 | train loss: 0.6347744968795777 train_acc: 0.78176 | val_loss: 0.6287663303375244 val_acc: 0.7925 
epoch: 20 | train loss: 0.4924998080253601 train_acc: 0.8324 | val_loss: 0.49918627309799196 val_acc: 0.8387 
epoch: 30 | train loss: 0.4287836483478546 train_acc: 0.8522 | val_loss: 0.4827147444725037 val_acc: 0.8473 
epoch: 40 | train loss: 0.3945864398765564 train_acc: 0.86194 | val_loss: 0.492375479221344 val_acc: 0.8502 
epoch: 50 | train loss: 0.3607987613296509 train_acc: 0.876 | val_loss: 0.4954912546157837 val_acc: 0.8489 
